# Load CARWatch logs and Study Manager results

This notebook demonstrates the package capabilities available after the raw-log and Study Manager loaders. It uses synthetic data from `examples/data` and does not access the ignored `playground` directory.

Register the project environment before opening the notebook with `uv run poe conf_jupyter`, then select the `carwatch` kernel.

In [ ]:
from pathlib import Path

from carwatch.io import load_logs, load_saliva, load_study_results
from carwatch.logs import extract_awakening_events_from_raw_logs, extract_sample_events_from_raw_logs, extract_day_summary_from_summary, extract_awakening_events_from_summary, extract_sample_events_from_summary
from carwatch.merge import merge_saliva

In [ ]:
DATA_DIR = Path("examples/data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")
DATA_DIR

## Raw app logs

`load_logs` parses semicolon-delimited app logs, converts Unix milliseconds into timezone-aware timestamps, and retains JSON payloads as dictionaries.

In [ ]:
logs = load_logs(
    DATA_DIR / "carwatch_demo_VP01_20250515.csv",
    tz="Europe/Berlin",
)
logs[["timestamp", "action", "source_file"]]

## Extract sampling and awakening events

The extraction functions retain the scheduled sample, the sample recorded by the app, and any mismatch between them.

In [ ]:
log_samples = extract_sample_events_from_raw_logs(logs)
log_awakening = extract_awakening_events_from_raw_logs(logs)

log_samples[[
    "sampling_time",
    "scheduled_sample",
    "recorded_sample",
    "sample_mismatch",
]]

In [ ]:
log_awakening

## Study Manager export

`load_study_results` retains one row per participant. Its column levels identify the day, sample, and variable without repeating day-level information for every sample.

In [ ]:
study_results = load_study_results(
    DATA_DIR / "study_results.csv",
    tz="Europe/Berlin",
)
study_results

## Focused Study Manager views

The helper functions extract day-level awakening information and sample-level observations only when those representations are needed. The original mismatch summary remains available once per day.

In [ ]:
study_awakening = extract_awakening_events_from_summary(study_results)
study_days = extract_day_summary_from_summary(study_results)
study_samples = extract_sample_events_from_summary(study_results)
study_awakening

In [ ]:
study_days

In [ ]:
study_samples

## Audit recorded sample swaps

The per-sample mismatch flag is derived from `scheduled_sample` and `recorded_sample`.

In [ ]:
mismatches = study_samples.loc[study_samples["sample_mismatch"].fillna(False)]
mismatches[["recorded_sample"]]

## Merge saliva measurements

The merge uses `recorded_sample` from the app. Swapped tubes are assigned to the scheduled sampling position at which they were actually collected. The `sample_id_source` column documents whether the assignment came from the app record or the study schedule.

In [ ]:
saliva = load_saliva(DATA_DIR / "saliva_samples.csv")
merged_samples = merge_saliva(study_samples, saliva)
merged_samples[[
    "sampling_time",
    "time_min",
    "recorded_sample",
    "cortisol",
    "sample_id_source",
    "mismatch_corrected",
]]